In [1]:
# Repository bootstrap: make local packages importable from any notebook working directory.
from pathlib import Path
import sys
_HERE = Path.cwd().resolve()
_REPO_ROOT = next((p for p in (_HERE, *_HERE.parents) if (p / 'publication_reconstruction').is_dir() and (p / 'data').is_dir()), None)
if _REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root from the notebook working directory.')
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))


# Section 11 — Unified Cross-Diagnostic Analysis

This notebook demonstrates one reusable cross-diagnostic workflow through two synthetic pairings: **Magnetic ↔ Hard X-ray** and **Electrostatic ↔ Turbulence**. Synthetic data are methodological demonstrations, not reconstructions of original IR-T1 measurements.

In [2]:
import sys
sys.path.insert(0, "../..")
from cross_diagnostic.synthetic import magnetic_hxr_pair, electrostatic_turbulence_pair
from cross_diagnostic.alignment import align_time_series
from cross_diagnostic.comparison import correlation_coefficient, cross_correlation, spectral_comparison
from cross_diagnostic.metadata import ExperimentalCondition
from turbulence_transport.transport_analysis.flux import turbulent_radial_particle_flux


## 1. Magnetic ↔ Hard X-ray

The two synthetic signals intentionally use different sampling rates. Alignment therefore requires an explicit target rate; the workflow never resamples silently.

In [3]:
magnetic, hxr = magnetic_hxr_pair()
magnetic_aligned, hxr_aligned = align_time_series(magnetic, hxr, target_sampling_frequency_hz=400_000)
zero_lag = correlation_coefficient(magnetic_aligned, hxr_aligned)
lags, corr, peak_lag, peak_corr = cross_correlation(magnetic_aligned, hxr_aligned)
spectrum = spectral_comparison(magnetic_aligned, hxr_aligned, nperseg=1024)
print("zero-lag correlation:", zero_lag)
print("maximum-correlation lag (s):", peak_lag)
print("correlation at that lag:", peak_corr)


zero-lag correlation: 0.7695275309107454
maximum-correlation lag (s): 0.0
correlation at that lag: 0.7695275309107449


A lag is reported as a temporal offset only. It is not automatically interpreted as causality.

## 2. Electrostatic ↔ Turbulence

The synthetic electrostatic potential fluctuation and turbulence density/velocity fluctuations share controlled temporal structure. The radial turbulent particle flux is calculated separately from the cross-diagnostic statistical comparison.

In [4]:
potential, density, radial_velocity = electrostatic_turbulence_pair()
zero_lag_density = correlation_coefficient(potential, density)
lags2, corr2, peak_lag2, peak_corr2 = cross_correlation(potential, density)
flux_r = turbulent_radial_particle_flux(density.data, radial_velocity.data)
print("potential-density zero-lag correlation:", zero_lag_density)
print("potential-density maximum-correlation lag (s):", peak_lag2)
print("synthetic radial turbulent particle flux (m^-2 s^-1):", flux_r)


potential-density zero-lag correlation: 0.9867258511920994
potential-density maximum-correlation lag (s): 0.0
synthetic radial turbulent particle flux (m^-2 s^-1): 4.7666927689546105e+19


## 3. Experimental-condition metadata

Experimental conditions remain separate from signal arrays and can be attached to analysis results.

In [5]:
condition = ExperimentalCondition(values={"hydrogen_pressure_Torr": 2.3, "limiter_bias_V": 200.0}, label="synthetic demonstration condition")
print(condition)


ExperimentalCondition(values={'hydrogen_pressure_Torr': 2.3, 'limiter_bias_V': 200.0}, label='synthetic demonstration condition')
